# 15a — An evidence-backed financial research pipeline — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/15.md) · [Course map](../PLAN.md) · [Project](../../projects/stage15/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Connect authorized retrieval and deterministic calculation
- Preserve provenance across steps
- Return explicit missing-data outcomes


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Connecting the evidence-backed pipeline

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

The capstone combines prior pieces into one request path: authenticated API, scoped evidence/tools, deterministic calculations, optional narrative drafting, and a response with sources and unknowns. Each boundary should have an explicit input/output contract.

End-to-end tests cross these boundaries. A correct arithmetic function alone does not prove the API uses the right tenant, year, source, or error behavior. Build a verified numerical table before asking a model to explain it.

A model narrative is initially a draft. Check each claim against the verified table and source text. Missing periods or unknown companies should lead to a defined unavailable/unknown response rather than silently invented zeros.

### Worked example: follow the values

For two companies and three years, expect six unique company/year rows. Verify each row's revenue, operating margin, free cash flow, source ID, and units. Repeat the request as the other tenant and ask for an unknown company and absent year. All are part of the contract.

### Pause and explain

Why should you test uniqueness as well as row count?

<details><summary>Check your reasoning after trying</summary>

Six rows could include a duplicate and omit one requested period. Verify the set of company/year keys too.

</details>

### Common mistakes to check

A prose claim can be wrong even when its numeric table is correct. Treat generated narrative validation as a separate stage.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

The capstone combines earlier components rather than hiding them behind one prompt. Authenticate the caller, scope the data, interpret a bounded request, retrieve source rows, calculate metrics in code, validate support, and return a report with citations. A local model can phrase a narrative, but it should not invent or silently recompute financial numbers.

Use typed requests and structured intermediate results so every step can be inspected. Record source IDs, company, period, currency, publication time, and calculation version. A missing year is a missing year, not zero revenue. Multi-company comparisons must align units, definitions, and periods before arithmetic.

The supplied reference includes a runnable local API and optional PostgreSQL/vector/model adapters. It is an instructional baseline; full capstone completion requires your independent implementation, broader curated evaluation, operational drills, and evidence that the deployed integrations work. A successful happy-path demo alone is insufficient.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import csv
rows = list(csv.DictReader((ROOT / "datasets" / "financials.csv").open()))
print(rows[0])


## Exercise 1: Period selection

Select requested company and years from rows, rejecting duplicate company-year records or missing years. Return chronological rows.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def select_periods(rows, company, years):
    selected = {}
    for row in rows:
        year = int(row["year"])
        if row["company"] == company and year in years:
            if year in selected: raise ValueError("Duplicate period")
            selected[year] = row
    if set(selected) != set(years): raise ValueError("Missing period")
    return [selected[year] for year in sorted(selected)]


In [ ]:
selected = select_periods(rows, "Aurora", [2023, 2024, 2025])
assert [int(r["year"]) for r in selected] == [2023, 2024, 2025]
expect_error(ValueError, lambda: select_periods(rows, "Aurora", [1999]))
print("Exercise 1: checks passed")


**Why this works:** Duplicate and missing periods should block a misleading comparison.


## Exercise 2: Verified financial row

Return company,year,revenue,operating_margin,fcf,source_id using Decimal inputs; revenue must be positive.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def calculate_row(row):
    from decimal import Decimal
    revenue = Decimal(row["revenue"])
    if revenue <= 0: raise ValueError("Invalid revenue")
    return {"company": row["company"], "year": int(row["year"]), "revenue": str(revenue), "operating_margin": str(Decimal(row["operating_income"])/revenue), "fcf": str(Decimal(row["operating_cash_flow"])-Decimal(row["capex"])), "source_id": row["source_id"]}


In [ ]:
result = calculate_row(select_periods(rows, "Aurora", [2025])[0])
assert result["revenue"] == "144" and result["fcf"] == "24"
assert result["source_id"] == "aurora-2025"
print("Exercise 2: checks passed")


**Why this works:** A number without its source and unit is difficult to audit.


## Exercise 3: Citation completeness

Return True when every report row has a nonempty source_id contained in allowed_source_ids.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def citations_complete(report, allowed_source_ids):
    return bool(report) and all(row.get("source_id") in allowed_source_ids for row in report)


In [ ]:
assert citations_complete([result], {"aurora-2025"})
assert not citations_complete([result], {"beacon-2025"})
assert not citations_complete([], set())
print("Exercise 3: checks passed")


**Why this works:** Source existence and permission are necessary; claim-level support still needs verification.


## Independent transfer

Build the two-company, three-year report through the API. Demonstrate citations, tenant isolation, unknown-company abstention, missing-period handling, and a local-model narrative constrained to verified numbers.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [15a interface and acceptance cases](../assignments/15a.md).
2. Copy the [blank starter](../assignments/starters/15a.py) to `work/assignments/15a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 15a --solution work/assignments/15a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/15a.md#15a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 15a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why compute outside the model?

   Deterministic arithmetic is easier to test, version, and audit.

2. What makes a comparison invalid?

   Mismatched periods, currencies, accounting definitions, or unavailable source evidence.


## Reading and review

- [Primary reference 1](https://fastapi.tiangolo.com/tutorial/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
